# GraphRAG Context Retrieval — HotpotQA Entity Graph Experiment

**Course:** 42913 Social and Information Network Analysis  
**Topic:** 2 — GraphRAG and Context Retrieval  

---

## Overview

This notebook demonstrates that the same graph-based scoring algorithms used on the
Wikipedia Vote Network also apply to a **knowledge-graph context retrieval** task.

**Dataset:** HotpotQA (Yang et al., 2018) — a multi-hop question answering benchmark.
Each question requires reasoning across 2 gold paragraphs surrounded by 8 noise paragraphs.

**Approach:**
1. Extract named entities from each paragraph using spaCy NER.
2. Build an **undirected entity co-occurrence graph** per question.
3. Seed each algorithm on **question entities** and rank all others by relevance score.
4. Label entities from **supporting facts and answers** as positive; all others negative.
5. Report **AUC-ROC** and **Average Precision** across 200 validation examples.

**Connection to main project:** The scoring functions are identical to those in
`src/graphrag/link_prediction.py`, confirming the `GraphRAGPipeline` generalises
beyond social networks to any entity graph.

## Setup

Install dependencies if running for the first time:
```
pip install datasets spacy scikit-learn networkx numpy
python -m spacy download en_core_web_sm
```

In [ ]:
import spacy
import networkx as nx
import numpy as np
from datasets import load_dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from collections import defaultdict

nlp = spacy.load("en_core_web_sm")
print("Dependencies loaded.")

Dependencies loaded.


## 1. Load Dataset — HotpotQA

In [ ]:
dataset = load_dataset("hotpot_qa", "distractor", split="validation[:500]")
# 'distractor' split: 2 gold paragraphs + 8 noise paragraphs per question
print(f"Loaded {len(dataset)} examples")
print(f"Features: {list(dataset.features.keys())}")

Loaded 500 examples
Features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']


In [ ]:
# Preview one example
ex = dataset[0]
print("Question :", ex['question'])
print("Answer   :", ex['answer'])
print("Support  :", ex['supporting_facts']['title'])
print(f"Context  : {len(ex['context']['title'])} paragraphs")

Question : Were Scott Derrickson and Ed Wood of the same nationality?
Answer   : yes
Support  : ['Scott Derrickson', 'Ed Wood']
Context  : 10 paragraphs


## 2. Graph Construction — Entity Co-occurrence

In [ ]:
ENTITY_LABELS = {"PERSON", "ORG", "GPE", "LOC", "WORK_OF_ART", "EVENT"}

def extract_entities(text: str) -> list[str]:
    """Extract named entities from text using spaCy NER."""
    doc = nlp(text)
    return [ent.text.lower().strip() for ent in doc.ents
            if ent.label_ in ENTITY_LABELS]

def build_example_graph(example: dict) -> nx.Graph:
    """
    Build a co-occurrence graph for one HotpotQA example.
    Nodes = named entities.
    Edge (u, v) = u and v appear in the same sentence.
    Edge weight = number of sentences they co-occur in.
    """
    G = nx.Graph()
    for _, sentences in zip(example["context"]["title"], example["context"]["sentences"]):
        for sentence in sentences:
            ents = list(set(extract_entities(sentence)))
            for i in range(len(ents)):
                for j in range(i + 1, len(ents)):
                    u, v = ents[i], ents[j]
                    if G.has_edge(u, v):
                        G[u][v]["weight"] += 1
                    else:
                        G.add_edge(u, v, weight=1)
    return G

G_example = build_example_graph(dataset[0])
print(f"Example graph: {G_example.number_of_nodes()} nodes, {G_example.number_of_edges()} edges")

Example graph: 24 nodes, 36 edges


## 3. Scoring Algorithms

In [ ]:
def score_candidates_ppr(G, question_entities, alpha=0.85):
    """Personalised PageRank seeded on question entities."""
    seeds = [e for e in question_entities if e in G.nodes]
    if not seeds or G.number_of_nodes() == 0:
        return {}
    personalization = {node: 0.0 for node in G.nodes}
    for s in seeds:
        personalization[s] = 1.0 / len(seeds)
    return nx.pagerank(G, alpha=alpha, personalization=personalization, weight='weight')

def score_candidates_jaccard(G, question_entities):
    """Jaccard coefficient between seed entities and all other nodes."""
    scores = defaultdict(float)
    seeds = [e for e in question_entities if e in G.nodes]
    for seed in seeds:
        for candidate in G.nodes:
            if candidate in seeds:
                continue
            try:
                j = list(nx.jaccard_coefficient(G, [(seed, candidate)]))[0][2]
                scores[candidate] = max(scores[candidate], j)
            except Exception:
                pass
    return dict(scores)

def score_candidates_adamic(G, question_entities):
    """Adamic/Adar index between seed entities and all other nodes."""
    scores = defaultdict(float)
    seeds = [e for e in question_entities if e in G.nodes]
    for seed in seeds:
        for u, v, s in nx.adamic_adar_index(G, [(seed, c) for c in G.nodes if c not in seeds]):
            scores[v] = max(scores[v], s)
    return dict(scores)

def score_candidates_katz(G, question_entities, beta=0.05, max_path=3):
    """Katz index: sums walk contributions of length 1..max_path, decayed by beta."""
    nodes = list(G.nodes)
    seeds = [e for e in question_entities if e in G.nodes]
    if not nodes or not seeds:
        return {}
    idx = {n: i for i, n in enumerate(nodes)}
    A = nx.to_numpy_array(G, nodelist=nodes, weight='weight')
    row_sums = A.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    A = A / row_sums
    seed_vec = np.zeros(len(nodes))
    for s in seeds:
        seed_vec[idx[s]] = 1.0 / len(seeds)
    score_vec = np.zeros(len(nodes))
    current = seed_vec.copy()
    for _ in range(max_path):
        current = beta * (A.T @ current)
        score_vec += current
    return {nodes[i]: score_vec[i] for i in range(len(nodes))}

print("All four scoring functions defined.")

All four scoring functions defined.


## 4. Evaluation Protocol

In [ ]:
def evaluate(dataset, scorer_fn, scorer_name, n_examples=200):
    """
    For each example: build the graph, score all entities from the question seed,
    label supporting-fact/answer entities as positive, compute AUC-ROC and Avg Precision.
    """
    all_labels, all_scores = [], []
    for example in list(dataset)[:n_examples]:
        G = build_example_graph(example)
        if G.number_of_nodes() < 5:
            continue
        question_ents = extract_entities(example['question'])
        positive_nodes = set(extract_entities(example['answer']))
        for title in example['supporting_facts']['title']:
            positive_nodes.update(extract_entities(title))
        positive_nodes = {e.lower().strip() for e in positive_nodes}
        scores = scorer_fn(G, question_ents)
        if not scores:
            continue
        for node, score in scores.items():
            all_scores.append(score)
            all_labels.append(1 if node in positive_nodes else 0)
    if sum(all_labels) == 0:
        print(f'{scorer_name}: no positives found')
        return
    auc = roc_auc_score(all_labels, all_scores)
    ap  = average_precision_score(all_labels, all_scores)
    print(f'{scorer_name:22s}  AUC-ROC: {auc:.4f}   Avg Precision: {ap:.4f}')

print("Evaluation function defined.")

Evaluation function defined.


## 5. Results — 200 HotpotQA Validation Examples

In [ ]:
print("Evaluating on 200 HotpotQA validation examples...")
print("-" * 60)
evaluate(dataset, score_candidates_ppr,     "Personalised PageRank")
evaluate(dataset, score_candidates_katz,    "Katz Index")
evaluate(dataset, score_candidates_jaccard, "Jaccard Coefficient")
evaluate(dataset, score_candidates_adamic,  "Adamic / Adar")
print("-" * 60)

Evaluating on 200 HotpotQA validation examples...
------------------------------------------------------------
Personalised PageRank    AUC-ROC: 0.8084   Avg Precision: 0.2228
Katz Index               AUC-ROC: 0.7252   Avg Precision: 0.0666
Jaccard Coefficient      AUC-ROC: 0.6295   Avg Precision: 0.0273
Adamic / Adar            AUC-ROC: 0.6158   Avg Precision: 0.0289
------------------------------------------------------------


## 6. Discussion and Conclusion

### Results Summary

| Method | AUC-ROC | Avg Precision |
|---|---:|---:|
| **Personalised PageRank** | **0.8084** | **0.2228** |
| Katz Index | 0.7252 | 0.0666 |
| Jaccard Coefficient | 0.6295 | 0.0273 |
| Adamic / Adar | 0.6158 | 0.0289 |
| Random baseline | ~0.500 | ~positives fraction |

### Key Findings

**Personalised PageRank achieves the highest AUC-ROC of 0.808**, confirming that
multi-hop graph traversal is the most effective strategy for context retrieval.
By seeding the random walk on question entities and following co-occurrence edges,
PPR propagates relevance to answer-supporting entities that may be several hops away.

**Katz Index ranks second (0.725)** because it explicitly sums walk contributions
across path lengths up to 3 hops, capturing more indirect structure than 1-hop methods.

**Jaccard and Adamic/Adar score similarly (~0.62)** because both are limited to
one-hop shared neighbours — insufficient for multi-hop question answering.

### Connection to the Main Project

The ranking **PPR > Katz > Jaccard ≈ Adamic/Adar** matches the pattern on the
Wikipedia Vote Network (PPR = 0.337 > CN = 0.303 > Jaccard = 0.294).
This consistency across two different graph types validates the `GraphRAGPipeline`
design — the same algorithms, the same interface, the same winner.